# U-Net++ Training


In [ ]:
import os
import numpy as np
import rasterio
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
from tensorflow.keras import layers, Model

def conv_block(x, filters, name=None):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=name+'_conv1')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=name+'_conv2')(x)
    return x


In [ ]:
def build_unetpp(input_shape=(256, 256, 3), filters=[64, 128, 256]):
    inputs = layers.Input(input_shape, name='input')

    # Encoder
    x00 = conv_block(inputs, filters[0], name='x00')
    p0 = layers.MaxPooling2D(pool_size=(2, 2))(x00)

    x10 = conv_block(p0, filters[1], name='x10')
    p1 = layers.MaxPooling2D(pool_size=(2, 2))(x10)

    x20 = conv_block(p1, filters[2], name='x20')

    # Decoder with nested skip connections
    x01 = conv_block(layers.Concatenate()([x00,
                                           layers.UpSampling2D(size=(2, 2))(x10)]),
                     filters[0], name='x01')

    x11 = conv_block(layers.Concatenate()([x10,
                                           layers.UpSampling2D(size=(2, 2))(x20)]),
                     filters[1], name='x11')

    x02 = conv_block(layers.Concatenate()([x00,
                                           x01,
                                           layers.UpSampling2D(size=(2, 2))(x11)]),
                     filters[0], name='x02')

    # Output layer
    output = layers.Conv2D(1, (1, 1), activation='sigmoid', name='final')(x02)

    return Model(inputs=inputs, outputs=output, name='unet_plus_plus')


In [ ]:
from tensorflow.keras import backend as K
import tensorflow as tf

# Define Dice Loss
def dice_loss(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Optionally combine with Binary Crossentropy
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    d_loss = dice_loss(y_true, y_pred)
    return bce + d_loss

def iou_metric(y_true, y_pred):
    y_pred = K.round(y_pred)
    y_true = K.cast(y_true, tf.float32)
    y_pred = K.cast(y_pred, tf.float32)

    intersection = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred) - intersection
    return intersection / (union + K.epsilon())

def dice_coefficient(y_true, y_pred):
    y_pred = K.round(y_pred)
    y_true = K.cast(y_true, tf.float32)
    y_pred = K.cast(y_pred, tf.float32)

    intersection = K.sum(y_true * y_pred)
    return (2. * intersection) / (K.sum(y_true) + K.sum(y_pred) + K.epsilon())

def f1_score(y_true, y_pred):
    y_pred = K.round(y_pred)
    y_true = K.cast(y_true, tf.float32)
    y_pred = K.cast(y_pred, tf.float32)

    tp = K.sum(y_true * y_pred)
    fp = K.sum((1 - y_true) * y_pred)
    fn = K.sum(y_true * (1 - y_pred))

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())

    return 2 * (precision * recall) / (precision + recall + K.epsilon())

In [ ]:
model.compile(
    optimizer='adam',
    loss=bce_dice_loss,  # <-- new loss function here
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        iou_metric,
        dice_coefficient,
        f1_score
    ]
)
cb = [
  tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
  tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
]

In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=40,
    batch_size=16,
    callbacks=cb
)

In [ ]:
# Loss
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Binary Crossentropy Loss')
plt.legend()
plt.show()
print("\n")

# Precision & Recall
plt.plot(history.history['precision'], label='prec_train')
plt.plot(history.history['val_precision'], label='prec_val')
plt.plot(history.history['recall'], label='rec_train')
plt.plot(history.history['val_recall'], label='rec_val')
plt.title('Precision & Recall')
plt.legend()
plt.show()
print("\n")

# Accuracy (if available)
if 'accuracy' in history.history and 'val_accuracy' in history.history:
    plt.plot(history.history['accuracy'], label='acc_train')
    plt.plot(history.history['val_accuracy'], label='acc_val')
    plt.title('Accuracy')
    plt.legend()
    plt.show()
    print("\n")

# F1 Score (if available)
if 'f1_score' in history.history and 'val_f1_score' in history.history:
    plt.plot(history.history['f1_score'], label='f1_train')
    plt.plot(history.history['val_f1_score'], label='f1_val')
    plt.title('F1 Score')
    plt.legend()
    plt.show()
    print("\n")

# IoU (if available)
if 'iou' in history.history and 'val_iou' in history.history:
    plt.plot(history.history['iou'], label='iou_train')
    plt.plot(history.history['val_iou'], label='iou_val')
    plt.title('Intersection over Union (IoU)')
    plt.legend()
    plt.show()
    print("\n")

# List all metrics tracked (keys in history.history)
metrics = list(history.history.keys())
print("Tracked metrics:")
print(metrics)

# Print final values for each metric
print("\nFinal epoch values:")
for metric in metrics:
    final_val = history.history[metric][-1]
    print(f"{metric}: {final_val:.4f}")


In [ ]:
idx = np.random.randint(0, X_val.shape[0])
pred = (model.predict(X_val[idx:idx+1])>0.5).astype(np.uint8)

fig,ax = plt.subplots(1,3,figsize=(12,4))
ax[0].imshow(X_val[idx]);      ax[0].set_title('Input')
ax[1].imshow(Y_val[idx].squeeze(),cmap='gray'); ax[1].set_title('True')
ax[2].imshow(pred.squeeze(),  cmap='gray');    ax[2].set_title('Predicted')
plt.show()


In [ ]:
# Save weights & model
model.save('cross_validate.keras')

In [ ]:
results = model.evaluate(X_test, Y_test, batch_size=2)
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

In [ ]:
import matplotlib.pyplot as plt

idx = 1  # change index to any valid test image

sample_img = X_test[idx]
sample_mask = Y_test[idx]
pred_mask = model.predict(np.expand_dims(sample_img, axis=0))[0]
pred_mask = (pred_mask > 0.5).astype(np.uint8)

# Plot the results
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(sample_img)
plt.title("Test Image")

plt.subplot(1, 3, 2)
plt.imshow(sample_mask.squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1, 3, 3)
plt.imshow(pred_mask.squeeze(), cmap='gray')
plt.title("Predicted Mask")
plt.show()
